# Install

In [1]:
%pip install geopandas shapely requests pyyaml pandas numpy

Note: you may need to restart the kernel to use updated packages.


# Setup

In [1]:
import pandas as pd
import numpy as np
import yaml
import geopandas as gpd
from shapely.geometry import Point
import re

# Load mmc data

In [2]:
path_to_file = "data/mmc-10.yaml"

In [3]:
with open(path_to_file, "r", encoding="utf-8") as file:
    raw_data = yaml.safe_load(file)
    
df = pd.DataFrame(raw_data)

In [4]:
df

,_id,url,topics,authors,date,figures,id,keywords,lead,mention,paragraphs,title,n_comments,gpt_keywords
0,673d34cb91440eaabc59f136,https://www.rtvslo.si/svet/vojna-v-ukrajini/mi...,svet,[B. V.],2024-11-19T15:30:02,[{'caption': 'Vojna v Ukrajini traja že 1000 d...,727977,"[Vojna, Sankcije, Rusija, Invazija, Ukrajina]",V vojni je bilo po podatkih ZN-a ubitih 12.164...,"[Dmitrij Peskov, Volodimir Zelenski, Vladimir ...","[""Ukrajina se ne bo nikoli podredila okupatorj...",Mineva 1000 dni od začetka invazije v Ukrajini,16.0,NaN
1,64cfe08650a608b5567fb4d2,https://www.rtvslo.si/slovenija/golob-rekordna...,slovenija,"[A. S., A. K. K.]",2023-08-04T13:15:42,"[{'caption': 'Marjan Šarec, Robert Golob in Sr...",677096,"[Robert Golob, Medsebojna pomoč, Predsednica d...",Današnja ujma je povzročila verjetno največjo ...,"[Marjan Šarec, Boštjan Poklukar, Srečko Šesta...","[""Po sobotni seji vlade bomo vložili novelo za...",Golob: Rekordna škoda zaradi ujm v samostojni ...,491.0,"[ujma, naravne nesreče, pomoč, premier, zakon,..."
2,686476be7500bfba23052f59,https://www.rtvslo.si/gospodarstvo/pogovori-me...,gospodarstvo,[T. L. Š.],2025-07-01T17:55:32,[{'caption': 'Kot so povedali v premierjevem k...,750667,"[Dialog, Zaprtje poslovalnic, Pogonska goriva,...",Pogovori med vlado in družbo Petrol se bodo po...,"[Robertom Golobom, Sašem Bergerjem, Aleksander...",[Današnji sestanek med Robertom Golobom in Saš...,Pogovori med vlado in Petrolom se bodo nadalje...,NaN,NaN
3,646b0b4ae7b5bc23d6763670,https://www.rtvslo.si/zabava-in-slog/znani/ana...,zabava-in-slog,[K. S.],2023-05-09T12:28:00,[{'caption': 'Bastian in Ana sta par od leta 2...,667491,"[Ana Ivanović, Bastian Schweinsteiger, Rojstvo]",Ana Ivanović in Bastian Schweinsteiger sta se ...,[],"[35-letna nekdanja srbska teniška zvezdnica, k...",Ana Ivanović in Bastian Schweinsteiger še tret...,4.0,"[Ana Ivanović, Bastian Schweinsteiger, otrok, ..."
4,66c7d13a11584f6db9b92754,https://www.rtvslo.si/gospodarstvo/cisti-dobic...,gospodarstvo,[G. K.],2024-08-22T17:47:24,[{'caption': 'Dobiček Luke Koper je bil v prve...,718765,"[Poslovni izid, Investicijski ciklus, Pretovor...",Luka Koper je v drugem letošnjem četrtletju na...,[],[V Luki Koper so nerevidirano polletno poročil...,Čisti dobiček Luke Koper ob polletju višji od ...,1.0,NaN
5,6672c87b1282e10945bb8252,https://www.rtvslo.si/sport/preostali-sporti/j...,sport,"[S. J., D. S. M.]",2024-06-19T12:22:21,[{'caption': 'Plavalka ravenskega Fužinarja Ja...,712298,"[plavanje, evropsko prvenstvo, Janja Šegel]",Na evropskem prvenstvu v plavanju smo videli d...,[Štafeta sedma],"[Šegel, ki bo v Beogradu v finalu nastopila že...","Janja Šegel verjame, da se bo na 200 m prosto ...",0.0,NaN
6,6645f57cd7b7a00dc0ed0bd9,https://www.rtvslo.si/zabava-in-slog/popkultur...,zabava-in-slog,[Ž. E. Č.],2024-05-16T09:17:27,"[{'caption': '""Modna revija Victoria's Secret ...",708416,"[Victoria's Secret, modna revija, vrnitev]",Po večletnem premoru se bodo jeseni supermanek...,"[Adriana Lima, Naomi Campbell, Bella Hadid, Ta...",[Victoria's Secret bo po več letih končno spet...,Vrača se legendarna modna revija Victoria's Se...,4.0,"[supermanekenke, krila, modna revija, Victoria..."
7,6467cd4d7db98a9226d45c8a,https://www.rtvslo.si/znanost-in-tehnologija/s...,znanost-in-tehnologija,[G. C.],2023-04-13T14:31:00,[{'img': 'https://img.rtvcdn.si/_up/upload/202...,664721,"[vesolje, ESA, MGRT, strategija]",Gospodarsko ministrstvo je predstavilo osnutek...,"[Matevž Frangež, Tanja Permozer, Gordon Campbell]","["" Slovenija je majhna na Zemlji, a želi posta...","""Slovenija je majhna na Zemlji, a želi postati...",13.0,"[vesolje, Slovenija, vesoljska industrija, ves..."
8,660685909bf5764acd04d86c,https://www.rtvslo.si/slovenija/premier-in-pos...,slovenija,[La. Da.],2024-03-13T11:28:00,"[{'caption': 'Foto: BoBo', 'img': 'https://img...",701404,"[Gibanje Svoboda, Robert Golob, poslanci]",Vodja poslanske skupine Gibanja Svoboda Borut ...,"[Boruta Sajovica, Petro Škofic, Mateja Arčona,...",[Poslans

# Load city locations

In [5]:
# Za določitev v kateri NUTS regiji je občina
url = "https://gisco-services.ec.europa.eu/distribution/v2/nuts/geojson/NUTS_RG_60M_2024_4326_LEVL_3.geojson"
gdf = gpd.read_file(url)
slovenia_regions = gdf[gdf['NUTS_ID'].str.startswith('SI', na=False)]

def get_slovenia_region(row):
    try:
        # Create the point
        point = Point(row['lng'], row['lat'])
        
        # Iterate through regions
        for _, region in slovenia_regions.iterrows():
            if region['geometry'].contains(point):
                return pd.Series([region['NUTS_NAME'], region['NUTS_ID']])
    except Exception:
        pass

    return pd.Series([None, None])

def slovensko_sklanjanje(row):
    ime = row["naselje"]
    if not isinstance(ime, str) or not ime:
        return pd.Series([None, None])

    # Pomožna funkcija za obdelavo posamezne besede
    def sklanjaj_besedo(beseda):
        # 1. Množinska imena (Abitanti, v Abitantih)
        if beseda.endswith('i'):
            return beseda + 'v', beseda + 'h'
        
        # 2. Ženska imena (Ljubljana, v Ljubljani)
        if beseda.endswith('a'):
            osnova = beseda[:-1]
            return osnova + 'e', osnova + 'i'
        
        # 3. Srednji spol (Velenje, Celje, Trebelno)
        if beseda.endswith('e') or (beseda.endswith('o') and len(beseda) > 3):
            osnova = beseda[:-1]
            # Preverimo prevoj (C, Č, Ž, Š, J) - v rodilniku ni vpliva, v mestniku pa
            return osnova + 'a', osnova + 'u'

        # 4. Moški spol (Maribor, Mokronog, Gradec)
        # Reševanje izpadajočega polglasnika (enostaven algoritem za -ec)
        osnova = beseda
        if beseda.endswith('ec'):
            osnova = beseda[:-2] + 'c'
        elif beseda.endswith('el') and beseda != 'Velenje': # npr. Angel -> Angla
            osnova = beseda[:-2] + 'l'
            
        return osnova + 'a', osnova + 'u'

    # Razbijemo na dele (upoštevamo presledke in vezaje)
    # Regex razbije "Mokronog-Trebelno" na ['Mokronog', '-', 'Trebelno']
    deli = re.split(r'(\s+|-)', ime)
    
    rodilnik_deli = []
    mestnik_deli = []
    
    for del_imena in deli:
        if del_imena.strip() == '' or del_imena == '-':
            rodilnik_deli.append(del_imena)
            mestnik_deli.append(del_imena)
        else:
            r, m = sklanjaj_besedo(del_imena)
            rodilnik_deli.append(r)
            mestnik_deli.append(m)
            
    return pd.Series(["".join(rodilnik_deli), "".join(mestnik_deli)])

In [7]:
# Za združitev občin z naselji
naselje_obcina_file = "data/naselje-obcina.csv"
obcina_geoloc_file = "data/obcina-geoloc.csv"

naselje_obcina = pd.read_csv(naselje_obcina_file, sep=";") 
obcina_geoloc = pd.read_csv(obcina_geoloc_file)

df_naselje_cord = pd.merge(
    naselje_obcina, 
    obcina_geoloc[['city', 'lat', 'lng']], 
    left_on='občina', 
    right_on='city', 
    how='left'
)
df_naselje_cord = df_naselje_cord.drop(columns=['city'])
df_naselje_cord = df_naselje_cord.drop(columns=['občina'])
df_naselje_cord = df_naselje_cord.dropna()

df_naselje_cord[["rodilnik", "mestnik"]] = df_naselje_cord.apply(slovensko_sklanjanje, axis=1)

# Poda regijo kordinati
df_naselje_cord[['region_name', 'region_id']] = df_naselje_cord.apply(get_slovenia_region, axis=1)

# Dropa random občine, ki so pre blizu borderja, da jim kordinati zajbavajo
df_naselje_cord = df_naselje_cord.dropna()

# Da so imena na začetku
cols = ["naselje", "rodilnik", "mestnik", "lat", "lng", "region_name", "region_id"]
df_naselje_cord = df_naselje_cord[cols]

In [11]:
cols

['naselje', 'rodilnik', 'mestnik', 'lat', 'lng', 'region_name', 'region_id']

In [18]:
import pandas as pd

pravilne_sklanjatve = [
    # ["naselje", "pravilen_rodilnik", "pravilen_mestnik"]
    ["Abitanti", "Abitantov", "Abitantih"],
    ["Adlešiči", "Adlešičev", "Adlešičih"],
    ["Adrijanci", "Adrijancev", "Adrijancih"],
    ["Arclin", "Arclina", "Arclinu"],
    ["Arja vas", "Arje vasi", "Arji vasi"],
    ["Artiče", "Artič", "Artičah"],
    ["Artmanja vas", "Artmanje vasi", "Artmanji vasi"],
    ["Babna Gorica", "Babne Gorice", "Babni Gorici"],
    ["Babna Gora", "Babne Gore", "Babni Gori"],
    ["Babni Vrt", "Babnega Vrta", "Babnem Vrtu"],
    ["Babiči", "Babičev", "Babičih"],
    ["Bač", "Bača", "Baču"],
    ["Bakovci", "Bakovcev", "Bakovcih"],
    ["Balkovci", "Balkovcev", "Balkovcih"],
    ["Banja Loka", "Banje Loke", "Banji Loki"],
    ["Banka", "Banke", "Banki"],
    ["Banjšice", "Banjšic", "Banjšicah"],
    ["Banovci", "Banovcev", "Banovcih"],
    ["Barislovci", "Barislovcev", "Barislovcih"],
    ["Bate", "Bat", "Batah"],
    ["Batiči", "Batičev", "Batičih"],
    ["Batuje", "Batuj", "Batujah"],
    ["Bavdek", "Bavdka", "Bavdku"],
    ["Begi", "Begov", "Begih"],
    ["Begunje pri Cerknici", "Begunj pri Cerknici", "Begunjah pri Cerknici"],
    ["Begunje na Gorenjskem", "Begunj na Gorenjskem", "Begunjah na Gorenjskem"],
    ["Bela Cerkev", "Bele Cerkve", "Beli Cerkvi"],
    ["Bela", "Bele", "Beli"],
    ["Belca", "Belce", "Belci"],
    ["Beli Grič", "Belega Griča", "Belem Griču"],
    ["Beli Potok pri Frankolovem", "Belega Potoka pri Frankolovem", "Belem Potoku pri Frankolovem"],
    ["Belo", "Belega", "Belem"],
    ["Belšinja vas", "Belšinje vasi", "Belšinji vasi"],
    ["Beltinci", "Beltincev", "Beltincih"],
    ["Beljevina", "Beljevine", "Beljevini"],
    ["Benetiči", "Benetičev", "Benetičih"],
    ["Benečija", "Benečije", "Benečiji"],
    ["Benošči", "Benoščev", "Benoščih"],
    ["Beričevo", "Beričevega", "Beričevem"],
    ["Besnica", "Besnice", "Besnici"],
    ["Betonovo", "Betonovega", "Betonovem"],
    ["Bevče", "Bevč", "Bevčah"],
    ["Bevke", "Bevk", "Bevkah"],
    ["Biba", "Bibe", "Bibi"],
    ["Bič", "Biča", "Biču"],
    ["Biljana", "Biljane", "Biljani"],
    ["Bilje", "Bilj", "Biljah"],
    ["Bimbas", "Bimbasa", "Bimbasu"],
    ["Binkelj", "Binklja", "Binklju"],
    ["Birčna vas", "Birčne vasi", "Birčni vasi"],
    ["Bisež", "Biseža", "Bisežu"],
    ["Bistra", "Bistre", "Bistri"],
    ["Bistrica", "Bistrice", "Bistrici"],
    ["Bistrica ob Dravi", "Bistrice ob Dravi", "Bistrici ob Dravi"],
    ["Bistrica ob Sotli", "Bistrice ob Sotli", "Bistrici ob Sotli"],
    ["Bistrica ob Sotli - del", "Bistrice ob Sotli", "Bistrici ob Sotli"],
    ["Bistrica pri Mariboru", "Bistrice pri Mariboru", "Bistrici pri Mariboru"],
    ["Bistričica", "Bistričice", "Bistričici"],
    ["Biš", "Biša", "Bišu"],
    ["Bišečki Vrh", "Bišečkega Vrha", "Bišečkem Vrhu"],
    ["Bišenci", "Bišencev", "Bišencih"],
    ["Bitnje", "Bitenj", "Bitnjah"],
    ["Bizeljska Vas", "Bizeljske Vasi", "Bizeljski Vasi"],
    ["Bizeljsko", "Bizeljskega", "Bizeljskem"],
    ["Blatna Brezovica", "Blatne Brezovice", "Blatni Brezovici"],
    ["Blatnik pri Črmošnjicah", "Blatnika pri Črmošnjicah", "Blatniku pri Črmošnjicah"],
    ["Blatnik v Podborštu", "Blatnika v Podborštu", "Blatniku v Podborštu"],
    ["Blato", "Blata", "Blatu"],
    ["Blečji Vrh", "Blečjega Vrha", "Blečjem Vrhu"],
    ["Bled", "Bleda", "Bledu"],
    ["Blejska Dobrava", "Blejske Dobrave", "Blejski Dobravi"],
    ["Blenkuš", "Blenkuša", "Blenkušu"],
    ["Bleskovica", "Bleskovice", "Bleskovici"],
    ["Blodnik", "Blodnika", "Blodniku"],
    ["Bločice", "Bločic", "Bločicah"],
    ["Bloška Polica", "Bloške Police", "Bloški Polici"],
    ["Boben", "Bobna", "Bobnu"],
    ["Bobovek", "Bobovka", "Bobovku"],
    ["Bobovišče", "Bobovišča", "Bobovišču"],
    ["Bocanjevci", "Bocanjevcev", "Bocanjevcih"],
    ["Bočna", "Bočne", "Bočni"],
    ["Bodislavci", "Bodislavcev", "Bodislavcih"],
    ["Bodkovci", "Bodkovcev", "Bodkovcih"],
    ["Bodonci", "Bodoncev", "Bodoncih"],
    ["Bodrež", "Bodreža", "Bodrežu"],
    ["Boginja vas", "Boginje vasi", "Boginji vasi"],
    ["Bogneča vas", "Bogneče vasi", "Bogneči vasi"],
    ["Bogo", "Boga", "Bogu"],
    ["Bogojina", "Bogojine", "Bogojini"],
    ["Boharina", "Boharine", "Boharini"],
    ["Bohinj", "Bohinja", "Bohinju"],
    ["Bohinjska Bela", "Bohinjske Bele", "Bohinjski Beli"],
    ["Bohinjska Bistrica", "Bohinjske Bistrice", "Bohinjski Bistrici"],
    ["Bohinjska Češnjica", "Bohinjske Češnjice", "Bohinjski Češnjici"],
    ["Bohor", "Bohorja", "Bohorju"],
    ["Bohova", "Bohove", "Bohovi"],
    ["Bojanci", "Bojancev", "Bojancih"],
    ["Bojana Vas", "Bojane Vasi", "Bojani Vasi"],
    ["Bojčići", "Bojčićev", "Bojčićih"],
    ["Bojna", "Bojne", "Bojni"],
    ["Bojnik", "Bojnika", "Bojniku"],
    ["Bojno", "Bojna", "Bojnu"],
    ["Bokalci", "Bokalcev", "Bokalcih"],
    ["Bokrači", "Bokračev", "Bokračih"],
    ["Boldraž", "Boldraža", "Boldražu"],
    ["Bolečka vas", "Bolečke vasi", "Bolečki vasi"],
    ["Bolešini", "Bolešinov", "Bolešinih"],
    ["Bolfenk", "Bolfenka", "Bolfenku"],
    ["Bolion", "Boliona", "Bolionu"],
    ["Boljunec", "Boljunca", "Boljuncu"],
    ["Bolko", "Bolka", "Bolku"],
    ["Bolska", "Bolske", "Bolski"],
    ["Bonini", "Boninov", "Boninih"],
    ["Borat", "Borata", "Boratu"],
    ["Boreci", "Borecev", "Borecih"],
    ["Boreča", "Boreče", "Boreči"],
    ["Borejci", "Borejcev", "Borejcih"],
    ["Boričevo", "Boričevega", "Boričevem"],
    ["Borjana", "Borjane", "Borjani"],
    ["Borje", "Borja", "Borju"],
    ["Borje pri Mlinšah", "Borja pri Mlinšah", "Borju pri Mlinšah"],
    ["Borji", "Borjev", "Borjih"],
    ["Bork", "Borka", "Borku"],
    ["Borovak pri Podkumu", "Borovaka pri Podkumu", "Borovaku pri Podkumu"],
    ["Borovak pri Polšniku", "Borovaka pri Polšniku", "Borovaku pri Polšniku"],
    ["Borovec", "Borovca", "Borovcu"],
    ["Borovec pri Kočevski Reki", "Borovca pri Kočevski Reki", "Borovcu pri Kočevski Reki"],
    ["Borovci", "Borovcev", "Borovcih"],
    ["Borovnica", "Borovnice", "Borovnici"],
    ["Borovniško", "Borovniškega", "Borovniškem"],
    ["Bortoli", "Bortolov", "Bortolih"],
    ["Boršt", "Boršta", "Borštu"],
    ["Boršt pri Dvoru", "Boršta pri Dvoru", "Borštu pri Dvoru"],
    ["Bosanska", "Bosanske", "Bosanski"],
    ["Bosie", "Bosij", "Bosijah"],
    ["Bosiljiva Loka", "Bosiljive Loke", "Bosiljivi Loki"],
    ["Bosljiva Loka", "Bosljive Loke", "Bosljivi Loki"],
    ["Boste", "Bost", "Bostah"],
    ["Bovec", "Bovca", "Bovcu"],
    ["Bovše", "Bovš", "Bovšah"],
    ["Božakovo", "Božakova", "Božakovu"],
    ["Božiči", "Božičev", "Božičih"],
    ["Božič Vrh", "Božičevega Vrha", "Božičevem Vrhu"],
    ["Božja", "Božje", "Božji"],
    ["Božje Brdo", "Božjega Brda", "Božjem Brdu"],
    ["Bračna vas", "Bračne vasi", "Bračni vasi"],
    ["Bradač", "Bradača", "Bradaču"],
    ["Brado", "Brada", "Bradu"],
    ["Brajdići", "Brajdićev", "Brajdičih"],
    ["Branik", "Branika", "Braniku"],
    ["Brankovci", "Brankovcev", "Brankovcih"],
    ["Branoslavci", "Branoslavcev", "Branoslavcih"],
    ["Brašnica", "Brašnice", "Brašnici"],
    ["Bratislavci", "Bratislavcev", "Bratislavcih"],
    ["Bratnice", "Bratnic", "Bratnicah"],
    ["Bratonci", "Bratoncev", "Bratoncih"],
    ["Bratonečice", "Bratonečic", "Bratonečicah"],
    ["Bratovš", "Bratovša", "Bratovšu"],
    ["Brce", "Brc", "Brcah"],
    ["Brda", "Brd", "Brdah"],
    ["Brdo", "Brda", "Brdu"],
    ["Brdo pri Lukovici", "Brda pri Lukovici", "Brdu pri Lukovici"],
    ["Brdce", "Brdc", "Brdcah"],
    ["Brdce nad Dobrno", "Brdc nad Dobrno", "Brdcih nad Dobrno"],
    ["Brecljevo", "Brecljevega", "Brecljevem"],
    ["Breg", "Brega", "Bregu"],
    ["Breg pri Borovnici", "Brega pri Borovnici", "Bregu pri Borovnici"],
    ["Breg pri Dobu", "Brega pri Dobu", "Bregu pri Dobu"],
    ["Breg pri Kočevju", "Brega pri Kočevju", "Bregu pri Kočevju"],
    ["Breg pri Komendi", "Brega pri Komendi", "Bregu pri Komendi"],
    ["Breg pri Konjicah", "Brega pri Konjicah", "Bregu pri Konjicah"],
    ["Breg pri Litiji", "Brega pri Litiji", "Bregu pri Litiji"],
    ["Breg pri Polzeli", "Brega pri Polzeli", "Bregu pri Polzeli"],
    ["Breg pri Ribnici na Dolenjskem", "Brega pri Ribnici na Dolenjskem", "Bregu pri Ribnici na Dolenjskem"],
    ["Breg pri Sinjem Vrhu", "Brega pri Sinjem Vrhu", "Bregu pri Sinjem Vrhu"],
    ["Breg v Loki", "Brega v Loki", "Bregu v Loki"],
    ["Brege", "Breg", "Bregah"],
    ["Breginj", "Breginja", "Breginju"],
    ["Brekovice", "Brekovic", "Brekovicah"],
    ["Brela", "Brel", "Brelah"],
    ["Bremšak", "Bremšaka", "Bremšaku"],
    ["Brendov", "Brendova", "Brendovu"],
    ["Bresnica", "Bresnice", "Bresnici"],
    ["Brest", "Bresta", "Brestu"],
    ["Brestanica", "Brestanice", "Brestanici"],
    ["Bresternica", "Bresternice", "Bresternici"],
    ["Breza", "Breze", "Brezi"],
    ["Brezari", "Brezarjev", "Brezarjih"],
    ["Brezje", "Brezja", "Brezju"],
    ["Brezje nad Kamnikom", "Brezja nad Kamnikom", "Brezju nad Kamnikom"],
    ["Brezje pod Nanosom", "Brezja pod Nanosom", "Brezju pod Nanosom"],
    ["Brezje pri Boštanju", "Brezja pri Boštanju", "Brezju pri Boštanju"],
    ["Brezje pri Dobrovi", "Brezja pri Dobrovi", "Brezju pri Dobrovi"],
    ["Brezje pri Dobu", "Brezja pri Dobu", "Brezju pri Dobu"],
    ["Brezje pri Grosupljem", "Brezja pri Grosupljem", "Brezju pri Grosupljem"],
    ["Brezje pri Kumpolju", "Brezja pri Kumpolju", "Brezju pri Kumpolju"],
    ["Brezje pri Lipoglavu", "Brezja pri Lipoglavu", "Brezju pri Lipoglavu"],
    ["Brezje pri Oplotnici", "Brezja pri Oplotnici", "Brezju pri Oplotnici"],
    ["Brezje pri Podplatu", "Brezja pri Podplatu", "Brezju pri Podplatu"],
    ["Brezje pri Poljčanah", "Brezja pri Poljčanah", "Brezju pri Poljčanah"],
    ["Brezje pri Raki", "Brezja pri Raki", "Brezju pri Raki"],
    ["Brezje pri Rožni Dolini", "Brezja pri Rožni Dolini", "Brezju pri Rožni Dolini"],
    ["Brezje pri Senušah", "Brezja pri Senušah", "Brezju pri Senušah"],
    ["Brezje pri Slovenski Bistrici", "Brezja pri Slovenski Bistrici", "Brezju pri Slovenski Bistrici"],
    ["Brezje pri Šentjerneju", "Brezja pri Šentjerneju", "Brezju pri Šentjerneju"],
    ["Brezje pri Trebelnem", "Brezja pri Trebelnem", "Brezju pri Trebelnem"],
    ["Brezje pri Tržiču", "Brezja pri Tržiču", "Brezju pri Tržiču"],
    ["Brezje pri Veliki Dolini", "Brezja pri Veliki Dolini", "Brezju pri Veliki Dolini"],
    ["Brezje v Podbočju", "Brezja v Podbočju", "Brezju v Podbočju"],
    ["Brezni Vrh", "Breznega Vrha", "Breznem Vrhu"],
    ["Breznica", "Breznice", "Breznici"],
    ["Breznica pod Lubnikom", "Breznice pod Lubnikom", "Breznici pod Lubnikom"],
    ["Breznica pri Žireh", "Breznice pri Žireh", "Breznici pri Žireh"],
    ["Breznik", "Breznika", "Brezniku"],
    ["Brezno", "Brezna", "Breznu"],
    ["Brezova Reber", "Brezove Reberi", "Brezovi Reberi"],
    ["Brezova Reber pri Dvoru", "Brezove Reberi pri Dvoru", "Brezovi Reberi pri Dvoru"],
    ["Brezovci", "Brezovcev", "Brezovcih"],
    ["Brezovdol", "Brezovdola", "Brezovdolu"],
    ["Brezovec", "Brezovca", "Brezovcu"],
    ["Brezovec pri Rogatcu", "Brezovca pri Rogatcu", "Brezovcu pri Rogatcu"],
    ["Brezovica", "Brezovice", "Brezovici"],
    ["Brezovica na Bizeljskem", "Brezovice na Bizeljskem", "Brezovici na Bizeljskem"],
    ["Brezovica pod Stolom", "Brezovice pod Stolom", "Brezovici pod Stolom"],
    ["Brezovica pri Borovnici", "Brezovice pri Borovnici", "Brezovici pri Borovnici"],
    ["Brezovica pri Črmošnjicah", "Brezovice pri Črmošnjicah", "Brezovici pri Črmošnjicah"],
    ["Brezovica pri Dobu", "Brezovice pri Dobu", "Brezovici pri Dobu"],
    ["Brezovica pri Gradinu", "Brezovice pri Gradinu", "Brezovici pri Gradinu"],
    ["Brezovica pri Ljubljana", "Brezovice pri Ljubljani", "Brezovici pri Ljubljani"],
    ["Brezovica pri Medvodah", "Brezovice pri Medvodah", "Brezovici pri Medvodah"],
    ["Brezovica pri Mirni", "Brezovice pri Mirni", "Brezovici pri Mirni"],
    ["Brezovica pri Stopičah", "Brezovice pri Stopičah", "Brezovici pri Stopičah"],
    ["Brezovica pri Trebelnem", "Brezovice pri Trebelnem", "Brezovici pri Trebelnem"],
    ["Brezovica pri Zlatem Polju", "Brezovice pri Zlatem Polju", "Brezovici pri Zlatem Polju"],
    ["Brezovica v Podbočju", "Brezovice v Podbočju", "Brezovici v Podbočju"],
    ["Brezovljani", "Brezovljanov", "Brezovljanih"],
    ["Brezovo", "Brezovega", "Brezovem"],
    ["Brezovska Gora", "Brezovske Gore", "Brezovski Gori"],
    ["Brezula", "Brezule", "Brezuli"],
    ["Briga", "Brige", "Brigi"],
    ["Brinje", "Brinja", "Brinju"],
    ["Brinjeca", "Brinjece", "Brinjeci"],
    ["Brinječka", "Brinječke", "Brinječki"],
    ["Brinjeva Gora", "Brinjeve Gore", "Brinjevi Gori"],
    ["Brink", "Brinka", "Brinku"],
    ["Briše", "Briš", "Brišah"],
    ["Briše pri Polhovem Gradcu", "Briš pri Polhovem Gradcu", "Brišah pri Polhovem Gradcu"],
    ["Briši", "Brišev", "Briših"],
    ["Britof", "Britofa", "Britofu"],
    ["Brje", "Brj", "Brjah"],
    ["Brje pri Koprivi", "Brj pri Koprivi", "Brjah pri Koprivi"],
    ["Brje pri Komnu", "Brj pri Komnu", "Brjah pri Komnu"],
    ["Brlog", "Brloga", "Brlogu"],
    ["Brlon", "Brlona", "Brlonu"],
    ["Brod", "Broda", "Brodu"],
    ["Brod v Podbočju", "Broda v Podbočju", "Brodu v Podbočju"],
    ["Brode", "Brod", "Brodah"],
    ["Bruna vas", "Brune vasi", "Bruni vasi"],
    ["Brunca", "Brunce", "Brunci"],
    ["Brunšvik", "Brunšvika", "Brunšviku"],
    ["Brusnice", "Brusnic", "Brusnicah"],
    ["Brust", "Brusta", "Brustu"],
    ["Brv", "Brvi", "Brvi"],
    ["Brvace", "Brvac", "Brvacah"],
    ["Brve", "Brv", "Brvah"],
    ["Brvnišče", "Brvnišča", "Brvnišču"],
    ["Bubnjarci", "Bubnjarcev", "Bubnjarcih"],
    ["Buc", "Buca", "Bucu"],
    ["Bucer", "Bucera", "Buceru"],
    ["Bucerji", "Bucerjev", "Bucerjih"],
    ["Bucika", "Bucike", "Buciki"],
    ["Buč", "Buča", "Buču"],
    ["Buče", "Buč", "Bučah"],
    ["Bučečovci", "Bučečovcev", "Bučečovcih"],
    ["Bučerca", "Bučerce", "Bučerci"],
    ["Buči", "Bučev", "Bučih"],
    ["Bučica", "Bučice", "Bučici"],
    ["Bučka", "Bučke", "Bučki"],
    ["Bučke-del", "Bučke", "Bučki"],
    ["Bučkovci", "Bučkovcev", "Bučkovcih"],
    ["Budanščica", "Budanščice", "Budanščici"],
    ["Budanja vas", "Budanje vasi", "Budanji vasi"],
    ["Budanje", "Budanj", "Budanjah"],
    ["Budganja vas", "Budganje vasi", "Budganji vasi"],
    ["Budenci", "Budencev", "Budencih"],
    ["Budganja Vas", "Budganje Vasi", "Budganji Vasi"],
    ["Budinci", "Budincev", "Budincih"],
    ["Budna vas", "Budne vasi", "Budni vasi"],
    ["Budni", "Budnih", "Budnih"],
    ["Buje", "Buj", "Bujah"],
    ["Bujani", "Bujanov", "Bujanih"],
    ["Bujanec", "Bujanca", "Bujancu"],
    ["Bujkovci", "Bujkovcev", "Bujkovcih"],
    ["Buko", "Buka", "Buku"],
    ["Bukor", "Bukorja", "Bukorju"],
    ["Bukov Vrh", "Bukovega Vrha", "Bukovem Vrhu"],
    ["Bukov Vrh e", "Bukovega Vrha", "Bukovem Vrhu"],
    ["Bukov Vrh e-del", "Bukovega Vrha", "Bukovem Vrhu"],
    ["Bukova Gora", "Bukove Gore", "Bukovi Gori"],
    ["Bukova Gora-del", "Bukove Gore", "Bukovi Gori"],
    ["Bukovec", "Bukovca", "Bukovcu"],
    ["Bukovec pri Poljanah", "Bukovca pri Poljanah", "Bukovcu pri Poljanah"],
    ["Bukovica", "Bukovice", "Bukovici"],
    ["Bukovica pri Litiji", "Bukovice pri Litiji", "Bukovici pri Litiji"],
    ["Bukovica pri Vodicah", "Bukovice pri Vodicah", "Bukovici pri Vodicah"],
    ["Bukovje", "Bukovja", "Bukovju"],
    ["Bukovje v Babni Gori", "Bukovja v Babni Gori", "Bukovju v Babni Gori"],
    ["Bukovlji", "Bukovljev", "Bukovljih"],
    ["Bukovnik", "Bukovnika", "Bukovniku"],
    ["Bukovska Vas", "Bukovske Vasi", "Bukovski Vasi"],
    ["Bukovščica", "Bukovščice", "Bukovščici"],
    ["Bukovje-del", "Bukovja", "Bukovju"],
    ["Bukovsko", "Bukovskega", "Bukovskem"],
    ["Bula", "Bule", "Buli"],
    ["Bulin", "Bulina", "Bulinu"],
    ["Bulina", "Buline", "Bulini"],
    ["Buline", "Bulin", "Bulinah"],
    ["Bunderji", "Bunderjev", "Bunderjih"],
    ["Bunčani", "Bunčanov", "Bunčanih"],
    ["Bunči", "Bunčev", "Bunčih"],
    ["Burd", "Burda", "Burdu"],
    ["Burda", "Burde", "Burdi"],
    ["Burdov", "Burdova", "Burdovu"],
    ["Burdu", "Burda", "Burdu"],
    ["Burg", "Burga", "Burgu"],
    ["Burga", "Burge", "Burgi"],
    ["Burgi", "Burgov", "Burgih"],
    ["Burgar", "Burgarja", "Burgarju"],
    ["Burja", "Burje", "Burji"],
    ["Burje", "Burj", "Burjah"],
    ["Burji", "Burjev", "Burjih"],
    ["Burjica", "Burjice", "Burjici"],
    ["Burjon", "Burjona", "Burjonu"],
    ["Bus", "Busa", "Busu"],
    ["Busa", "Buse", "Busi"],
    ["Busan", "Busana", "Busanu"],
    ["Busani", "Busanov", "Busanih"],
    ["Busen", "Busena", "Busenu"],
    ["Busenc", "Busenca", "Busencu"],
    ["Busenci", "Busencev", "Busencih"],
    ["Busi", "Busov", "Busih"],
    ["Busica", "Busice", "Busici"],
    ["Busije", "Busij", "Busijah"],
    ["Busik", "Busika", "Busiku"],
    ["Busiki", "Busikov", "Busikih"],
    ["Busin", "Busina", "Businu"],
    ["Butari", "Butarjev", "Butarjih"],
    ["Butajna", "Butajne", "Butajni"],
    ["Butajnova", "Butajnove", "Butajnovi"],
    ["Butale", "Butal", "Butalah"],
    ["Butali", "Butalov", "Butalih"],
    ["Butan", "Butana", "Butanu"],
    ["Butari-del", "Butarjev", "Butarjih"],
    ["Butge", "Butg", "Butgah"],
    ["Buti", "Butov", "Butih"],
    ["Butič", "Butiča", "Butiču"],
    ["Butiči", "Butičev", "Butičih"],
    ["Butkovci", "Butkovcev", "Butkovcih"],
    ["Butkovišče", "Butkovišča", "Butkovišču"],
    ["Butle", "Butl", "Butlah"],
    ["Butli", "Butlov", "Butlih"],
    ["Butmil", "Butmila", "Butmilu"],
    ["Butmile", "Butmil", "Butmilah"],
    ["Butmili", "Butmilov", "Butmilih"],
    ["Butmilska", "Butmilske", "Butmilski"],
    ["Butmir", "Butmira", "Butmiru"],
    ["Butmira", "Butmire", "Butmiri"],
    ["Butmire", "Butmir", "Butmirah"],
    ["Butmiri", "Butmirov", "Butmirih"],
    ["Butmirsko", "Butmirskega", "Butmirskem"],
    ["Butn", "Butna", "Butnu"],
    ["Butna", "Butne", "Butni"],
    ["Butnar", "Butnarja", "Butnarju"],
    ["Butnarji", "Butnarjev", "Butnarjih"],
    ["Butne", "Butn", "Butnah"],
    ["Butni", "Butnov", "Butnih"],
    ["Butnica", "Butnice", "Butnici"],
    ["Butnice", "Butnic", "Butnicah"],
    ["Butnici", "Butnicov", "Butnicih"],
    ["Butnik", "Butnika", "Butniku"],
    ["Butniki", "Butnikov", "Butnikih"],
    ["Butnov", "Butnova", "Butnovu"],
    ["Butnovo", "Butnovega", "Butnovem"],
    ["Buton", "Butona", "Butonu"],
    ["Butona", "Butone", "Butoni"],
    ["Butoniga", "Butonige", "Butonigi"],
    ["Butonige", "Butonig", "Butonigah"],
    ["Butonigi", "Butonigov", "Butonigih"],
    ["Butor", "Butorja", "Butorju"],
    ["Butora", "Butore", "Butori"],
    ["Butoraj", "Butoraja", "Butoraju"],
    ["Butoraje", "Butoraj", "Butorajah"],
    ["Butoraji", "Butorajev", "Butorajih"],
    ["Butorajn", "Butorajna", "Butorajnu"],
    ["Butorajne", "Butorajn", "Butorajnah"],
    ["Butorajni", "Butorajnov", "Butorajnih"],
    ["Butorajska", "Butorajske", "Butorajski"],
    ["Butorajsko", "Butorajskega", "Butorajskem"],
    ["Butoras", "Butorasa", "Butorasu"],
    ["Butorase", "Butoras", "Butorasah"],
    ["Butorasi", "Butorasov", "Butorasih"],
    ["Butore", "Butor", "Butorah"],
    ["Butori", "Butorjev", "Butorjih"],
    ["Butorica", "Butorice", "Butorici"],
    ["Butorice", "Butoric", "Butoricah"],
    ["Butorici", "Butoricov", "Butoricih"],
    ["Butorin", "Butorina", "Butorinu"],
    ["Butorina", "Butorine", "Butorini"],
    ["Butorine", "Butorin", "Butorinah"],
    ["Butorini", "Butorinov", "Butorinih"],
    ["Butorinski", "Butorinskega", "Butorinskem"],
    ["Butorinsko", "Butorinskega", "Butorinskem"],
    ["Butork", "Butorka", "Butorku"],
    ["Butorka", "Butorke", "Butorki"],
    ["Butorke", "Butork", "Butorkah"],
    ["Butorki", "Butorkov", "Butorkih"],
    ["Butorn", "Butorna", "Butornu"],
    ["Butorna", "Butorne", "Butorni"],
    ["Butorne", "Butorn", "Butornah"],
    ["Butorni", "Butornov", "Butornih"],
    ["Butorov", "Butorova", "Butorovu"],
    ["Butorova", "Butorove", "Butorovi"],
    ["Butorove", "Butorov", "Butorovah"],
    ["Butorovi", "Butorovov", "Butorovih"],
    ["Butorovo", "Butorovega", "Butorovem"],
    ["Butorska", "Butorske", "Butorski"],
    ["Butorsko", "Butorskega", "Butorskem"],
    ["Cankova", "Cankove", "Cankovi"],
    ["Celje", "Celja", "Celju"],
    ["Cerklje na Gorenjskem", "Cerkelj na Gorenjskem", "Cerkljah na Gorenjskem"],
    ["Cerklje ob Krki", "Cerkelj ob Krki", "Cerkljah ob Krki"],
    ["Cerknica", "Cerknice", "Cerknici"],
    ["Cerkno", "Cerkna", "Cerknu"],
    ["Cirkulane", "Cirkulan", "Cirkulanah"],
    ["Črenšovci", "Črenšovcev", "Črenšovcih"],
    ["Črna na Koroškem", "Črne na Koroškem", "Črni na Koroškem"],
    ["Črnomelj", "Črnomlja", "Črnomlju"],
    ["Destrnik", "Destrnika", "Destrniku"],
    ["Divača", "Divače", "Divači"],
    ["Dobje pri Planini", "Dobja pri Planini", "Dobju pri Planini"],
    ["Dobrepolje", "Dobrepolja", "Dobrepolju"],
    ["Dobrna", "Dobrne", "Dobrni"],
    ["Dobrova-Polhov Gradec", "Dobrove-Polhovega Gradca", "Dobrovi-Polhovem Gradcu"],
    ["Dobrovnik", "Dobrovnika", "Dobrovniku"],
    ["Dolenjske Toplice", "Dolenjskih Toplic", "Dolenjskih Toplicah"],
    ["Domžale", "Domžal", "Domžalah"],
    ["Dornava", "Dornave", "Dornavi"],
    ["Dramlje", "Dramelj", "Dramljah"],
    ["Dravograd", "Dravograda", "Dravogradu"],
    ["Duplek", "Dupleka", "Dupleku"],
    ["Gorenja vas-Poljane", "Gorenje vasi-Poljan", "Gorenji vasi-Poljanah"],
    ["Gorišnica", "Gorišnice", "Gorišnici"],
    ["Gorje", "Gorij", "Gorjah"],
    ["Gornja Radgona", "Gornje Radgone", "Gornji Radgoni"],
    ["Gornji Grad", "Gornjega Grada", "Gornjem Gradu"],
    ["Gornji Petrovci", "Gornjih Petrovcev", "Gornjih Petrovcih"],
    ["Gradišče", "Gradišča", "Gradišču"],
    ["Grosuplje", "Grosupljega", "Grosupljem"],
    ["Hajdina", "Hajdine", "Hajdini"],
    ["Hoče-Slivnica", "Hoč-Slivnice", "Hočah-Slivnici"],
    ["Hodoš", "Hodoša", "Hodošu"],
    ["Horjul", "Horjula", "Horjulu"],
    ["Hrastnik", "Hrastnika", "Hrastniku"],
    ["Hrpelje-Kozina", "Hrpelj-Kozine", "Hrpeljah-Kozini"],
    ["Idrija", "Idrije", "Idriji"],
    ["Ig", "Iga", "Igu"],
    ["Ilirska Bistrica", "Ilirske Bistrice", "Ilirski Bistrici"],
    ["Ivančna Gorica", "Ivančne Gorice", "Ivančni Gorici"],
    ["Izola", "Izole", "Izoli"],
    ["Jesenice", "Jesenic", "Jesenicah"],
    ["Jezersko", "Jezerskega", "Jezerskem"],
    ["Juršinci", "Juršincev", "Juršincih"],
    ["Kamnik", "Kamnika", "Kamniku"],
    ["Kanal ob Soči", "Kanala ob Soči", "Kanalu ob Soči"],
    ["Kidričevo", "Kidričevega", "Kidričevem"],
    ["Kobarid", "Kobarida", "Kobaridu"],
    ["Kobilje", "Kobilja", "Kobilju"],
    ["Kočevje", "Kočevja", "Kočevju"],
    ["Komen", "Komna", "Komnu"],
    ["Komenda", "Komende", "Komendi"],
    ["Koper", "Kopra", "Kopru"],
    ["Kostanjevica na Krki", "Kostanjevice na Krki", "Kostanjevici na Krki"],
    ["Kostel", "Kostela", "Kostelu"],
    ["Kozje", "Kozjega", "Kozjem"],
    ["Kranj", "Kranja", "Kranju"],
    ["Kranjska Gora", "Kranjske Gore", "Kranjski Gori"],
    ["Križevci", "Križevcev", "Križevcih"],
    ["Krško", "Krškega", "Krškem"],
    ["Kungota", "Kungote", "Kungoti"],
    ["Kuzma", "Kuzme", "Kuzmi"],
    ["Laško", "Laškega", "Laškem"],
    ["Lenart", "Lenarta", "Lenartu"],
    ["Lendava", "Lendave", "Lendavi"],
    ["Litija", "Litije", "Litiji"],
    ["Ljubno", "Ljubnega", "Ljubnem"],
    ["Ljutomer", "Ljutomera", "Ljutomeru"],
    ["Log-Dragomer", "Loga-Dragomerja", "Logu-Dragomerju"],
    ["Logatec", "Logatca", "Logatcu"],
    ["Loška dolina", "Loške doline", "Loški dolini"],
    ["Loški Potok", "Loškega Potoka", "Loškem Potoku"],
    ["Lovrenc na Pohorju", "Lovrenca na Pohorju", "Lovrencu na Pohorju"],
    ["Luče", "Luč", "Lučah"],
    ["Lukovica", "Lukovice", "Lukovici"],
    ["Majšperk", "Majšperka", "Majšperku"],
    ["Makole", "Makol", "Makolah"],
    ["Maribor", "Maribora", "Mariboru"],
    ["Markovci", "Markovcev", "Markovcih"],
    ["Medvode", "Medvod", "Medvodah"],
    ["Mengeš", "Mengša", "Mengšu"],
    ["Metlika", "Metlike", "Metliki"],
    ["Mežica", "Mežice", "Mežici"],
    ["Miklavž na Dravskem polju", "Miklavža na Dravskem polju", "Miklavžu na Dravskem polju"],
    ["Miren-Kostanjevica", "Mirna-Kostanjevice", "Mirnu-Kostanjevici"],
    ["Mirna", "Mirne", "Mirni"],
    ["Mirna Peč", "Mirne Peči", "Mirni Peči"],
    ["Mislinja", "Mislinje", "Mislinji"],
    ["Mokronog-Trebelno", "Mokronoga-Trebelnega", "Mokronogu-Trebelnem"],
    ["Moravče", "Moravč", "Moravčah"],
    ["Moravske Toplice", "Moravskih Toplic", "Moravskih Toplicah"],
    ["Mozirje", "Mozirja", "Mozirju"],
    ["Murska Sobota", "Murske Sobote", "Murski Soboti"],
    ["Muta", "Mute", "Muti"],
    ["Naklo", "Naklega", "Naklem"],
    ["Nazarje", "Nazarja", "Nazarju"],
    ["Nova Gorica", "Nove Gorice", "Novi Gorici"],
    ["Novo mesto", "Novega mesta", "Novem mestu"],
    ["Odranci", "Odrancev", "Odrancih"],
    ["Oplotnica", "Oplotnice", "Oplotnici"],
    ["Ormož", "Ormoža", "Ormožu"],
    ["Osilnica", "Osilnice", "Osilnici"],
    ["Pesnica", "Pesnice", "Pesnici"],
    ["Piran", "Pirana", "Piranu"],
    ["Pivka", "Pivke", "Pivki"],
    ["Podčetrtek", "Podčetrtka", "Podčetrtku"],
    ["Podlehnik", "Podlehnika", "Podlehniku"],
    ["Podvelka", "Podvelke", "Podvelki"],
    ["Poljčane", "Poljčan", "Poljčanah"],
    ["Polzela", "Polzele", "Polzeli"],
    ["Postojna", "Postojne", "Postojni"],
    ["Prebold", "Prebolda", "Preboldu"],
    ["Preddvor", "Preddvora", "Preddvoru"],
    ["Prevalje", "Prevalj", "Prevaljah"],
    ["Ptuj", "Ptuja", "Ptuju"],
    ["Puconci", "Puconcev", "Puconcih"],
    ["Rače-Fram", "Rač-Frama", "Račah-Framu"],
    ["Radeče", "Radeč", "Radečah"],
    ["Radenci", "Radencev", "Radencih"],
    ["Radlje ob Dravi", "Radelj ob Dravi", "Radljah ob Dravi"],
    ["Radovljica", "Radovljice", "Radovljici"],
    ["Ravne na Koroškem", "Raven na Koroškem", "Ravnah na Koroškem"],
    ["Razkrižje", "Razkrižja", "Razkrižju"],
    ["Rečica ob Savinji", "Rečice ob Savinji", "Rečici ob Savinji"],
    ["Renče-Vogrsko", "Renč-Vogrskega", "Renčah-Vogrskem"],
    ["Ribnica na Pohorju", "Ribnice na Pohorju", "Ribnici na Pohorju"],
    ["Ribnica", "Ribnice", "Ribnici"],
    ["Rogaška Slatina", "Rogaške Slatine", "Rogaški Slatini"],
    ["Rogašovci", "Rogašovcev", "Rogašovcih"],
    ["Rogatec", "Rogatca", "Rogatcu"],
    ["Ruše", "Ruš", "Rušah"],
    ["Selnica ob Dravi", "Selnice ob Dravi", "Selnici ob Dravi"],
    ["Sevnica", "Sevnice", "Sevnici"],
    ["Sežana", "Sežane", "Sežani"],
    ["Slovenj Gradec", "Slovenj Gradca", "Slovenj Gradcu"],
    ["Slovenska Bistrica", "Slovenske Bistrice", "Slovenski Bistrici"],
    ["Slovenske Konjice", "Slovenske Konjice", "Slovenskih Konjicah"],
    ["Sodražica", "Sodražice", "Sodražici"],
    ["Solčava", "Solčave", "Solčavi"],
    ["Središče ob Dravi", "Središča ob Dravi", "Središču ob Dravi"],
    ["Starše", "Starš", "Staršah"],
    ["Straža", "Straže", "Straži"],
    ["Sveta Ana", "Svete Ane", "Sveti Ani"],
    ["Sveta Trojica v Slovenskih goricah", "Svete Trojice v Slovenskih goricah", "Sveti Trojici v Slovenskih goricah"],
    ["Sveti Andraž v Slovenskih goricah", "Svetega Andraža v Slovenskih goricah", "Svetem Andražu v Slovenskih goricah"],
    ["Sveti Jurij ob Ščavnici", "Svetega Jurija ob Ščavnici", "Svetem Juriju ob Ščavnici"],
    ["Sveti Jurij v Slovenskih goricah", "Svetega Jurija v Slovenskih goricah", "Svetem Juriju v Slovenskih goricah"],
    ["Sveti Tomaž", "Svetega Tomaža", "Svetem Tomažu"],
    ["Šalovci", "Šalovcev", "Šalovcih"],
    ["Šempeter-Vrtojba", "Šempetra-Vrtojbe", "Šempetru-Vrtojbi"],
    ["Šenčur", "Šenčurja", "Šenčurju"],
    ["Šentilj", "Šentilja", "Šentilju"],
    ["Šentjernej", "Šentjerneja", "Šentjerneju"],
    ["Šentjur", "Šenturja", "Šenturju"],
    ["Šentrupert", "Šentruperta", "Šentrupertu"],
    ["Škocjan", "Škocjana", "Škocjanu"],
    ["Škofja Loka", "Škofje Loke", "Škofji Loki"],
    ["Škofljica", "Škofljice", "Škofljici"],
    ["Šmarje pri Jelšah", "Šmarja pri Jelšah", "Šmarju pri Jelšah"],
    ["Šmarješke Toplice", "Šmarjeških Toplic", "Šmarjeških Toplicah"],
    ["Šmartno ob Paki", "Šmartna ob Paki", "Šmartnem ob Paki"],
    ["Šmartno pri Litiji", "Šmartna pri Litiji", "Šmartnem pri Litiji"],
    ["Šoštanj", "Šoštanja", "Šoštanju"],
    ["Štore", "Štor", "Štorah"],
    ["Tabor", "Tabora", "Taboru"],
    ["Tišina", "Tišine", "Tišini"],
    ["Tolmin", "Tolmina", "Tolminu"],
    ["Trbovlje", "Trbovelj", "Trbovljah"],
    ["Trebnje", "Trebnjega", "Trebnjem"],
    ["Trnovska vas", "Trnovske vasi", "Trnovski vasi"],
    ["Trzin", "Trzina", "Trzinu"],
    ["Tržič", "Tržiča", "Tržiču"],
    ["Turnišče", "Turnišča", "Turnišču"],
    ["Velenje", "Velenja", "Velenju"],
    ["Velika Polana", "Velike Polane", "Veliki Polani"],
    ["Velike Lašče", "Velikih Lašč", "Velikih Laščah"],
    ["Veržej", "Veržeja", "Veržeju"],
    ["Videm", "Vidma", "Vidmu"],
    ["Vipava", "Vipave", "Vipavi"],
    ["Vitanje", "Vitanja", "Vitanju"],
    ["Vodice", "Vodic", "Vodicah"],
    ["Vojnik", "Vojnika", "Vojniku"],
    ["Vransko", "Vranskega", "Vranskem"],
    ["Vrhnika", "Vrhnike", "Vrhniki"],
    ["Vuzenica", "Vuzenice", "Vuzenici"],
    ["Zagorje ob Savi", "Zagorja ob Savi", "Zagorju ob Savi"],
    ["Zavrč", "Zavrča", "Zavrču"],
    ["Zreče", "Zreč", "Zrečah"],
    ["Žalec", "Žalca", "Žalcu"],
    ["Železniki", "Železnikov", "Železnikih"],
    ["Žetale", "Žetal", "Žetalah"],
    ["Žiri", "Žirov", "Žireh"],
    ["Žirovnica", "Žirovnice", "Žirovnici"],
    ["Žužemberk", "Žužemberka", "Žužemberku"],
    ["Žuniči", "Žuničev", "Žuničih"],
]

# 2. Pretvorba v DataFrame za posodobitev tvojega df_naselje_cord
df_popravki = pd.DataFrame(
    pravilne_sklanjatve, columns=["naselje", "rodilnik_new", "mestnik_new"]
)

# 3. Združevanje in zamenjava vrednosti v df_naselje_cord
df_naselje_cord = pd.merge(
    df_naselje_cord, df_popravki, on="naselje", how="left"
)
df_naselje_cord["rodilnik"] = df_naselje_cord["rodilnik_new"].fillna(
    df_naselje_cord["rodilnik"]
)
df_naselje_cord["mestnik"] = df_naselje_cord["mestnik_new"].fillna(
    df_naselje_cord["mestnik"]
)

# Brisanje pomožnih stolpcev
df_naselje_cord = df_naselje_cord.drop(
    columns=["rodilnik_new", "mestnik_new"]
)

In [19]:
df_naselje_cord

,naselje,rodilnik,mestnik,lat,lng,region_name,region_id
0,Abitanti,Abitantov,Abitantih,45.5500,13.7333,Obalno-kraška,SI044
1,Adamovo,Adamova,Adamovu,45.8363,14.6377,Osrednjeslovenska,SI041
2,Adlešiči,Adlešičev,Adlešičih,45.5711,15.1889,Jugovzhodna Slovenija,SI037
3,Adrijanci,Adrijancev,Adrijancih,46.8050,16.2172,Pomurska,SI031
4,Ajba,Ajbe,Ajbi,46.0880,13.6347,Goriška,SI043
...,...,...,...,...,...,...,...
4947,Žurkov Dol,Žurkova Dola,Žurkovu Dolu,46.0092,15.3041,Posavska,SI036
4948,Žužemberk,Žužemberka,Žužemberku,45.8339,14.9292,Jugovzhodna Slovenija,SI037
4949,Žvab,Žvaba,Žvabu,46.4086,16.1475,Podravska,SI032
4950,Žvabovo,Žvabova,Žvabovu,45.8389,15.3361,Jugovzhodna Slovenija,SI037


In [ ]:
#4952

naselje        4952
rodilnik       4952
mestnik        4952
lat            4952
lng            4952
region_name    4952
region_id      4952
dtype: int64

In [ ]:
# Odstranbe
# Krka ker podjetje
bannedBesede = [ "tabor", "kot", "krog", "konec", "svet", "vrh", "ravni", "liga", "svetu", "vrata",
    "koncu", "KRKA", "ter", "vrhu", "okrog", "rob", "sedlo", "vir", "Jeruzalem", "nemci",
    "meja", "ravno", "železnice", "križ", "plače", "pogled", "starše", "hudo", "srednje",
    "ladja", "površju", "globoko", "ceste", "gradnja", "naredi", "Kralji", "gola", "anže",
    "občina", "jesen", "prazniki", "bele", "vojska", "vrt", "bela", "anže", "apače", "bat",
    "bela", "bele", "belo", "bič", "bistra", "bistre", "blata", "blato", "blatno", "blatu",
    "bogo", "boršt", "brd", "brda", "breg", "breg", "brezen", "brlog", "buče", "cente",
    "cesta", "cola", "dana", "dol", "draga", "gaj", "golo", "gora", "gore", "gorenje",
    "dolenje", "gozd", "gradec", "gradbišče", "gradišče", "grm", "gruča", "hlebce", "hrib",
    "hudo", "jama", "javor", "jezero", "kamenje", "kanal", "klada", "koče", "konca", "konj",
    "kopriva", "korita", "koti", "kozje", "križ", "križate", "lom", "lopar", "lopata",
    "luža", "luže", "mački", "meja", "občine", "orla", "otok", "paradiž", "peč", "pekel",
    "planina", "plat", "ples", "pleša", "potok", "ravne", "razdrto", "repa", "rim", "rob",
    "sava", "slap", "soča", "srednje", "stanu", "straža", "suho", "suha", "suša", "sveto",
    "škala", "tlaka", "travnik", "trata", "vas", "vaš", "vaše", "vir", "vrba", "vrata",
    "vrh", "vrt", "zagon", "zastava", "železno", "župa",
    "polje", "pušča", "postaja", "raka", "huje", "ledina", "laže", "pobegi", "imeno",
    "primož", "obrat", "okroglo", "smreka", "cerkev"]
for beseda in bannedBesede:
	df_naselje_cord = df_naselje_cord.query(
    'naselje.str.lower() != @beseda.lower() and '
    'rodilnik.str.lower() != @beseda.lower() and '
    'mestnik.str.lower() != @beseda.lower()'
	)

In [23]:
# Tu lahko preveriš, če je beseda res odstranjena
beseda = "rob"
df_naselje_cord.query(
    'naselje.str.lower() == @beseda.lower() or '
    'rodilnik.str.lower() == @beseda.lower() or '
    'mestnik.str.lower() == @beseda.lower()'
	)

,naselje,rodilnik,mestnik,lat,lng,region_name,region_id


In [24]:
print(df_naselje_cord.head())
print(df_naselje_cord.count())

     naselje    rodilnik     mestnik      lat      lng            region_name  \
0   Abitanti   Abitantov   Abitantih  45.5500  13.7333          Obalno-kraška   
1    Adamovo     Adamova     Adamovu  45.8363  14.6377      Osrednjeslovenska   
2   Adlešiči   Adlešičev   Adlešičih  45.5711  15.1889  Jugovzhodna Slovenija   
3  Adrijanci  Adrijancev  Adrijancih  46.8050  16.2172               Pomurska   
4       Ajba        Ajbe        Ajbi  46.0880  13.6347                Goriška   

  region_id  
0     SI044  
1     SI041  
2     SI037  
3     SI031  
4     SI043  
naselje        4723
rodilnik       4723
mestnik        4723
lat            4723
lng            4723
region_name    4723
region_id      4723
dtype: int64


In [25]:
df_naselje_cord.to_csv("./processed_data/naselja.csv")